# Оптимизаторы: подробный разбор и сравнение

Здесь собраны формулы, варианты адаптивных методов и современные матричные оптимизаторы. Основная цепочка — SGD, momentum, Нестеров, AdaGrad, RMSprop, Adam и AdamW. Далее — методы экономии памяти и матричного предобусловливания.

Математические примеры выполняются при обычном запуске. Сравнение CNN на CIFAR-10 включают переменной `W11_OPT_CIFAR=1` до запуска ядра.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from IPython.display import display
from support.week11_utils import load_data, show_images, show_table

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

from support.lesson11_visuals import setup_style
setup_style()

In [ ]:
from support.week11_utils import draw_matrices, draw_pipeline, plot_metric_panels

torch.set_num_threads(4)

## Механизм, стоимость и условия сравнения

**Оптимизатор**, или solver, задает преобразование градиента в изменение параметров.
Слово «лучше» требует критерия: меньше шагов до данного loss, меньше секунд,
меньше памяти или выше качество на новых данных. Эти критерии могут расходиться.

Последовательность основного разбора: SGD → momentum → Нестеров → AdaGrad → RMSprop → Adam.
Для каждого метода сначала показывают формулу и опыт на общей функции
$f(w)=(w_0^2+20w_1^2)/2$ с одним стартом $(3,2)$.
Заданные шаги в иллюстрациях различаются и **не служат честным рейтингом**.
Затем каждому дают одинаковое число попыток подбора и выполняют контрольные опыты.
Современные матричные методы разбирают на матрицах, а не выдают двумерный пример
за доказательство их качества на языковых моделях.

In [ ]:
import copy
import os
import time
from pathlib import Path

from support.week11_utils import (
    ImageDataset, benchmark_inference, channel_statistics, choose_device,
    classification_metrics, evaluate_model, make_loader, paired_accuracy_interval,
    plot_histories, predict_probabilities, seed_everything, synchronize,
)

# Одна поверхность для всех наглядных примеров: минимум в (0,0).
# rotation меняет ориентацию осей, но сохраняет собственные значения 1 и 20.
def optimizer_objective(point, rotation=0.0):
    if point.ndim == 2:
        point = point.diagonal()
    cosine = np.cos(rotation)
    sine = np.sin(rotation)
    first = cosine * point[0] + sine * point[1]
    second = -sine * point[0] + cosine * point[1]
    value = (first.square() + 20 * second.square()) / 2
    return value


def trace_solver(point, solver, steps=100, rotation=0.0, noise_std=0.0, seed=10):
    generator = np.random.default_rng(seed)
    gradient_noise = generator.normal(0, noise_std, size=(steps, 2))
    coordinates = point.diagonal() if point.ndim == 2 else point
    positions = [coordinates.detach().numpy().copy()]
    losses = [optimizer_objective(point, rotation).item()]
    update_norms = []
    state_history = []
    started = time.perf_counter()
    for step in range(steps):
        solver.zero_grad(set_to_none=True)
        loss = optimizer_objective(point, rotation)
        noise = torch.tensor(gradient_noise[step], dtype=point.dtype)
        coordinates = point.diagonal() if point.ndim == 2 else point
        surrogate = loss + (noise * coordinates).sum()
        surrogate.backward()
        previous = point.detach().clone()
        solver.step()
        update_norms.append((point.detach() - previous).norm().item())
        coordinates = point.diagonal() if point.ndim == 2 else point
        positions.append(coordinates.detach().numpy().copy())
        losses.append(optimizer_objective(point, rotation).item())
        state = solver.state.get(point, {})
        copied = {}
        for key, value in state.items():
            if torch.is_tensor(value):
                copied[key] = value.detach().numpy().copy()
        state_history.append(copied)
        if not np.isfinite(losses[-1]) or losses[-1] > 1e12:
            break
    result = {"positions": np.asarray(positions), "losses": np.asarray(losses),
              "update_norms": np.asarray(update_norms), "states": state_history,
              "rotation": rotation, "seconds": time.perf_counter() - started}
    return result


from support.optimizer_demo import plot_solver_trace

solver_traces = {}

## 1. SGD: общий шаг и шум мини-батча

Для батча $B_t$ градиент $g_t=|B_t|^{-1}\sum_{i\in B_t}\nabla\ell_i(w_t)$,
обновление $w_{t+1}=w_t-\eta g_t$. На полной функции градиент детерминирован;
на разных батчах появляется шум. Ниже сначала используют полный градиент одной
квадратичной функции, затем в общем сравнении добавляют одинаковый шум всем методам.

Память состояния без momentum — практически нулевая. В узкой долине один общий шаг
ограничен направлением с наибольшей кривизной: при Hessian с собственными значениями
1 и 20 устойчивость требует $0<\eta<2/20$. Это ограничивает прогресс и вдоль пологой оси.

In [ ]:
point = nn.Parameter(torch.tensor([3., 2.], dtype=torch.float64))
solver = torch.optim.SGD([point], lr=0.03)
trace = trace_solver(point, solver, steps=100)
solver_traces["SGD"] = trace

## 2. Momentum: накопление направления

Сначала вычисляют $b_t=\mu b_{t-1}+g_t$, затем $w_{t+1}=w_t-\eta b_t$.
Подстановка предыдущего шага дает $b_t=g_t+\mu g_{t-1}+\mu^2g_{t-2}+\ldots$.
Согласованные компоненты накапливаются, меняющие знак частично компенсируются.

Это может ускорить движение вдоль пологой оси, но накопленная скорость вызывает
перелет и колебания. Большое значение momentum не гарантирует улучшения.
Хранят один дополнительный тензор формы параметров. В примере шаг тот же, что у SGD.

In [ ]:
point = nn.Parameter(torch.tensor([3., 2.], dtype=torch.float64))
solver = torch.optim.SGD([point], lr=0.03, momentum=0.9)
trace = trace_solver(point, solver, steps=100)
solver_traces["Momentum"] = trace

## 3. Нестеров: градиент после предсказанного сдвига

В записи со скоростью $v$ сначала предсказывают точку
$\tilde w_t=w_t+\mu v_{t-1}$, затем вычисляют
$v_t=\mu v_{t-1}-\eta\nabla L(\tilde w_t)$ и $w_{t+1}=w_t+v_t$.
Градиент корректирует уже ожидаемый инерционный сдвиг, поэтому торможение
может начинаться раньше, чем у обычного momentum.

PyTorch использует эквивалентную сдвинутую параметризацию при постоянном шаге:
$b_t=\mu b_{t-1}+g_t$, $d_t=g_t+\mu b_t$, $\theta_{t+1}=\theta_t-\eta d_t$,
где $\theta_t$ соответствует точке предсказания. Промежуточные $w_t$ двух записей
не совпадают; в `SGD(nesterov=True)` отдельного второго `forward` нет.
Метод хранит один буфер, как momentum; шаг также требует настройки.

In [ ]:
point = nn.Parameter(torch.tensor([3., 2.], dtype=torch.float64))
solver = torch.optim.SGD([point], lr=0.03, momentum=0.9, nesterov=True)
trace = trace_solver(point, solver, steps=100)
solver_traces["Nesterov"] = trace

In [ ]:
# Учебная запись с реальным заглядыванием вперед; скорости заданы явно.
current = np.array([1., .8])
velocity = np.array([-.3, -.4])
mu = .9
eta = .03
lookahead = current + mu * velocity
gradient_here = np.array([current[0], 20 * current[1]])
gradient_ahead = np.array([lookahead[0], 20 * lookahead[1]])
momentum_next = lookahead - eta * gradient_here
nesterov_next = lookahead - eta * gradient_ahead
figure, axis = plt.subplots(figsize=(8, 4.5))
# Та же функция: уровни помогают сопоставить инерционный сдвиг и поправку.
nag_w0 = np.linspace(-.15, 1.15, 180)
nag_w1 = np.linspace(-.15, .95, 180)
nag_grid_w0, nag_grid_w1 = np.meshgrid(nag_w0, nag_w1)
nag_surface = (nag_grid_w0 ** 2 + 20 * nag_grid_w1 ** 2) / 2
nag_levels = axis.contour(
    nag_grid_w0, nag_grid_w1, nag_surface,
    levels=[.05, .2, .5, 1, 2, 4, 6, 8],
    colors="#c3c8cd", linewidths=.8, zorder=0,
)
axis.clabel(nag_levels, levels=[.2, 1, 4], fmt="%g", fontsize=7, colors="#7b838c")
axis.set_aspect("equal", adjustable="box")
axis.scatter(0, 0, color="black", marker="*", s=70, label="минимум", zorder=3)
for destination, title, color in [(lookahead, "предсказанный сдвиг", "grey"),
                                  (momentum_next, "momentum: градиент здесь", "darkred"),
                                  (nesterov_next, "Нестеров: градиент впереди", "#0072CE")]:
    axis.annotate("", xy=destination, xytext=current, arrowprops={"arrowstyle": "->", "color": color})
    axis.scatter(*destination, color=color, label=title)
axis.scatter(*current, color="black", label="текущая точка")
axis.axhline(0, color="silver")
axis.set(xlabel="w0", ylabel="w1", title="Инерция одинакова; место оценки градиента различается")
axis.legend(fontsize=8)
figure.tight_layout()
plt.show()

# Проверка именно формулы PyTorch, без смешения двух параметризаций.
nag_parameter = nn.Parameter(torch.tensor([1., .8], dtype=torch.float64))
nag_solver = torch.optim.SGD([nag_parameter], lr=.03, momentum=.9, nesterov=True)
buffer = torch.zeros_like(nag_parameter)
for step in range(4):
    nag_solver.zero_grad(set_to_none=True)
    optimizer_objective(nag_parameter).backward()
    gradient = nag_parameter.grad.detach().clone()
    buffer = .9 * buffer + gradient
    expected = nag_parameter.detach().clone() - .03 * (gradient + .9 * buffer)
    nag_solver.step()
    assert torch.allclose(nag_parameter, expected)

## 4. AdaGrad: накопленная сумма квадратов

Для каждой координаты $s_{t,j}=s_{t-1,j}+g_{t,j}^2$,
$w_{t+1,j}=w_{t,j}-\eta g_{t,j}/(\sqrt{s_{t,j}}+\varepsilon)$.
Большие или частые градиенты увеличивают знаменатель этой координаты.
Редко обновляемые признаки сохраняют относительно большой эффективный шаг.

Поскольку складываются неотрицательные числа, $s_t$ не убывает и эффективный шаг
постепенно уменьшается. Это полезно для некоторых разреженных задач, но может
слишком рано замедлить долгий стационарный опыт. Хранят один тензор состояния.

In [ ]:
point = nn.Parameter(torch.tensor([3., 2.], dtype=torch.float64))
solver = torch.optim.Adagrad([point], lr=0.3, eps=1e-10)
trace = trace_solver(point, solver, steps=100)
solver_traces["AdaGrad"] = trace

## 5. RMSprop: забывание старых квадратов

Вместо бесконечной суммы используют экспоненциальное среднее:
$s_t=\rho s_{t-1}+(1-\rho)g_t^2$,
$w_{t+1}=w_t-\eta g_t/(\sqrt{s_t}+\varepsilon)$.
Вклад старого градиента через $k$ шагов уменьшается в $\rho^k$ раз.

Знаменатель адаптируется к текущему масштабу, а не растет только из-за возраста
обучения. В отличие от Adam, в базовом RMSprop нет первого момента и поправки
на начальное смещение второго момента. В PyTorch momentum и centered-режим
включаются отдельно; здесь оба выключены.

In [ ]:
point = nn.Parameter(torch.tensor([3., 2.], dtype=torch.float64))
solver = torch.optim.RMSprop([point], lr=0.03, alpha=0.9, momentum=0, centered=False)
trace = trace_solver(point, solver, steps=100)
solver_traces["RMSprop"] = trace

In [ ]:
# Воспроизводят историю одного градиента: всплеск закончился, затем g=1.
gradient_stream = np.concatenate([np.full(10, 10.), np.ones(90)])
adagrad_cache = np.cumsum(gradient_stream ** 2)
rms_cache = []
cache = 0.0
for gradient in gradient_stream:
    cache = .9 * cache + .1 * gradient ** 2
    rms_cache.append(cache)
rms_cache = np.array(rms_cache)
figure, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(adagrad_cache, label="AdaGrad: сумма")
axes[0].plot(rms_cache, label="RMSprop: EMA")
axes[0].set(ylabel="накопленный квадрат", xlabel="шаг", title="Старая большая величина: помнить или забыть")
axes[1].plot(1 / (np.sqrt(adagrad_cache) + 1e-8), label="AdaGrad")
axes[1].plot(1 / (np.sqrt(rms_cache) + 1e-8), label="RMSprop")
axes[1].set(ylabel="множитель при общем eta", xlabel="шаг", title="Разные эффективные шаги при том же g")
for axis in axes:
    axis.legend()
    axis.grid(alpha=.2)
figure.tight_layout()
plt.show()

## 6. Adam: два момента и поправка на нулевой старт

Используют $m_t=\beta_1m_{t-1}+(1-\beta_1)g_t$ и
$v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2$.
При постоянном градиенте $g$ и нулевом старте $m_t=(1-\beta_1^t)g$;
деление на $1-\beta_1^t$ убирает это начальное занижение. Аналогично для $v_t$:

$$\hat m_t=\frac{m_t}{1-\beta_1^t},\quad
\hat v_t=\frac{v_t}{1-\beta_2^t},\quad
w_{t+1}=w_t-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\varepsilon}.$$

Первый момент сглаживает направление, второй задает покоординатный масштаб.
Это не отменяет подбор $\eta$: чрезмерный шаг остается чрезмерным.
Память — два тензора формы параметров, без учета самих весов и градиентов.

In [ ]:
point = nn.Parameter(torch.tensor([3., 2.], dtype=torch.float64))
solver = torch.optim.Adam([point], lr=0.15, betas=(0.9, 0.999))
trace = trace_solver(point, solver, steps=100)
solver_traces["Adam"] = trace

In [ ]:
# Постоянный градиент позволяет проверить поправку аналитически.
beta1, beta2 = .9, .999
steps = np.arange(1, 31)
moment = 1 - beta1 ** steps
second_moment = 1 - beta2 ** steps
corrected_moment = moment / (1 - beta1 ** steps)
corrected_second = second_moment / (1 - beta2 ** steps)
figure, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(steps, moment, label="m без поправки")
axes[0].plot(steps, corrected_moment, label="m с поправкой")
axes[1].plot(steps, second_moment, label="v без поправки")
axes[1].plot(steps, corrected_second, label="v с поправкой")
for axis in axes:
    axis.set(xlabel="шаг", ylabel="оценка момента", title="Постоянный g=1; истинное значение момента - 1")
    axis.legend()
    axis.grid(alpha=.2)
figure.tight_layout()
plt.show()

## 7. Дополнение: AdaDelta

AdaDelta хранит средние квадратов градиентов $E[g^2]_t$ и обновлений $E[\Delta w^2]_t$.
Из отношения их корней получают масштаб направления:
$u_t=-\sqrt{E[\Delta w^2]_{t-1}+\varepsilon}\,g_t/\sqrt{E[g^2]_t+\varepsilon}$.
Затем обновляют $E[\Delta w^2]_t$ квадратом $u_t$.

В PyTorch дополнительно применяют множитель `lr`: $w_{t+1}=w_t+\eta u_t$.
Идея уменьшает зависимость от внешнего масштаба шага, но не делает метод
универсальной современной заменой AdamW. Это историческое дополнение к основной цепочке.

In [ ]:
point = nn.Parameter(torch.tensor([3., 2.], dtype=torch.float64))
solver = torch.optim.Adadelta([point], lr=1.0, rho=0.9)
trace = trace_solver(point, solver, steps=100)
solver_traces["Adadelta"] = trace

## 8. Дополнение: Adamax, NAdam, RAdam и AMSGrad

Это разные модификации, а не четыре названия одного метода:

- Adamax заменяет второй момент на $u_t=\max(\beta_2u_{t-1},|g_t|+\varepsilon)$;
  знаменатель связан с экспоненциально взвешенной бесконечной нормой;
- NAdam добавляет Nesterov-подобную коррекцию первого момента; реализация PyTorch
  учитывает также расписание коэффициента momentum;
- RAdam корректирует дисперсию адаптивного шага в начале обучения;
- AMSGrad хранит покоординатный максимум вторых моментов, чтобы знаменатель
  не уменьшался произвольно; это вариант `Adam(amsgrad=True)`.

Ни один вариант не гарантирует лучшего качества на каждом наборе данных.
В примерах ниже меняют только оптимизатор на той же функции.

In [ ]:
point = nn.Parameter(torch.tensor([3., 2.], dtype=torch.float64))
solver = torch.optim.Adamax([point], lr=0.15)
trace = trace_solver(point, solver, steps=100)
solver_traces["Adamax"] = trace

### NAdam: отдельный численный запуск

Используют тот же старт, ту же функцию и сто обновлений; настройка видна в конструкторе.

In [ ]:
point = nn.Parameter(torch.tensor([3., 2.], dtype=torch.float64))
solver = torch.optim.NAdam([point], lr=.1)
trace = trace_solver(point, solver, steps=100)
solver_traces["NAdam"] = trace

### RAdam: отдельный численный запуск

Используют тот же старт, ту же функцию и сто обновлений; настройка видна в конструкторе.

In [ ]:
point = nn.Parameter(torch.tensor([3., 2.], dtype=torch.float64))
solver = torch.optim.RAdam([point], lr=.1)
trace = trace_solver(point, solver, steps=100)
solver_traces["RAdam"] = trace

### AMSGrad: отдельный численный запуск

Используют тот же старт, ту же функцию и сто обновлений; настройка видна в конструкторе.

In [ ]:
point = nn.Parameter(torch.tensor([3., 2.], dtype=torch.float64))
solver = torch.optim.Adam([point], lr=.1, amsgrad=True)
trace = trace_solver(point, solver, steps=100)
solver_traces["AMSGrad"] = trace

## 9. AdamW: регуляризация отдельно от нормировки градиента

У Adam со штрафом $\lambda\|w\|^2/2$ к градиенту добавляется $\lambda w$;
эта сумма попадает и в $m$, и в $v$. Поэтому штраф также делится на адаптивный знаменатель.
AdamW применяет отдельно $w\leftarrow(1-\eta\lambda)w$, затем адаптивный шаг по градиенту данных.
При $\lambda=0$ и совпадающих остальных настройках обычные Adam и AdamW эквивалентны.
В коде это сначала проверяют, затем включают decay и показывают различие.

AdamW — практический базовый вариант для многих Transformer-рецептов;
например, он указан в техническом отчете DeepSeek-V3. Это факт конкретного рецепта,
а не доказательство превосходства на CIFAR-10 или доли на рынке оптимизаторов.

In [ ]:
# Adam без weight decay и AdamW без weight decay должны совпадать.
parameter_adam = nn.Parameter(torch.tensor([2., -1.], dtype=torch.float64))
parameter_adamw = nn.Parameter(parameter_adam.detach().clone())
plain_adam = torch.optim.Adam([parameter_adam], lr=.03, weight_decay=0)
plain_adamw = torch.optim.AdamW([parameter_adamw], lr=.03, weight_decay=0)
for step in range(20):
    for parameter, solver in [(parameter_adam, plain_adam), (parameter_adamw, plain_adamw)]:
        solver.zero_grad(set_to_none=True)
        optimizer_objective(parameter).backward()
        solver.step()
assert torch.allclose(parameter_adam, parameter_adamw, atol=1e-12)

# При ненулевом decay показывают различие самих обновлений на одних градиентах.
trajectories_decay = {}
for name, optimizer_class in [("Adam + L2", torch.optim.Adam), ("AdamW", torch.optim.AdamW)]:
    parameter = nn.Parameter(torch.tensor([2., -1.], dtype=torch.float64))
    solver = optimizer_class([parameter], lr=.03, weight_decay=.1)
    records = [parameter.detach().numpy().copy()]
    for step in range(60):
        solver.zero_grad(set_to_none=True)
        optimizer_objective(parameter).backward()
        solver.step()
        records.append(parameter.detach().numpy().copy())
    trajectories_decay[name] = np.asarray(records)
figure, axes = plt.subplots(1, 2, figsize=(11, 3.5))
# Фон - линии уровня потери данных f, общей для обоих методов.
decay_w0 = np.linspace(-.2, 2.2, 200)
decay_w1 = np.linspace(-1.2, .2, 200)
decay_grid_w0, decay_grid_w1 = np.meshgrid(decay_w0, decay_w1)
decay_loss = (decay_grid_w0 ** 2 + 20 * decay_grid_w1 ** 2) / 2
decay_levels = axes[0].contour(
    decay_grid_w0, decay_grid_w1, decay_loss,
    levels=[.05, .2, .5, 1, 2, 5, 10, 15],
    colors="#c3c8cd", linewidths=.8, zorder=0,
)
axes[0].clabel(decay_levels, levels=[.5, 2, 10], fmt="%g", fontsize=7, colors="#7b838c")
axes[0].set_aspect("equal", adjustable="box")

for name, positions in trajectories_decay.items():
    axes[0].plot(positions[:, 0], positions[:, 1], label=name)
    axes[1].plot(np.linalg.norm(positions, axis=1), label=name)
axes[0].set(xlabel="w0", ylabel="w1", title="Одно начало, одинаковые eta и lambda")
axes[1].set(xlabel="шаг", ylabel="||w||", title="L2 внутри Adam и decoupled decay различаются")
for axis in axes:
    axis.legend()
    axis.grid(alpha=.2)
figure.tight_layout()
plt.show()

## 10. Adafactor: факторизация ради памяти

Для матрицы второго момента $V\in\mathbb R^{r\times c}$ хранят средние строк
$a_i=c^{-1}\sum_jV_{ij}$ и столбцов $b_j=r^{-1}\sum_iV_{ij}$.
Восстанавливают $\tilde V_{ij}=a_ib_j/\operatorname{mean}(a)$.
Вместо $rc$ чисел получают $r+c$; это аппроксимация, не тождество.
Adafactor также использует относительный шаг, масштаб параметров и clipping обновления.

Метод опубликован в 2018 году и использован, в частности, в T5: он не «новее AdamW»,
но остается полезным представителем экономии состояния. В `torch.optim.Adafactor`
`lr` ограничивает относительный шаг; его нельзя трактовать как тот же абсолютный
множитель, что в SGD. Ниже отдельно показывают факторизацию и реальный API.

In [ ]:
# Механизм факторизации второго момента: хранить средние строк и столбцов.
generator = np.random.default_rng(21)
squared_gradients = np.exp(generator.normal(size=(12, 20)))
row_means = squared_gradients.mean(axis=1)
column_means = squared_gradients.mean(axis=0)
factorized_second = row_means[:, None] * column_means[None, :] / row_means.mean()
relative_error = np.linalg.norm(factorized_second - squared_gradients) / np.linalg.norm(squared_gradients)
draw_matrices([squared_gradients, factorized_second, squared_gradients - factorized_second],
              ["Полный второй момент", "Факторизованная аппроксимация", f"Ошибка, относительная норма {relative_error:.2f}"],
              annotate=False)

# Проверяют реальный состав состояния torch.optim.Adafactor.
afactor_parameter = nn.Parameter(torch.ones(12, 20))
afactor_solver = torch.optim.Adafactor([afactor_parameter], lr=.01, weight_decay=0)
afactor_parameter.grad = torch.tensor(np.sqrt(squared_gradients), dtype=torch.float32)
afactor_solver.step()
state_shapes = {name: tuple(value.shape) for name, value in afactor_solver.state[afactor_parameter].items() if torch.is_tensor(value)}
print("Реальное состояние Adafactor:", state_shapes)

rows, columns = 512, 2048
state_sizes = {"Adam: m и v": 2 * rows * columns * 4,
               "Adafactor: строка + столбец": (rows + columns) * 4}
plt.bar(state_sizes.keys(), np.array(list(state_sizes.values())) / 2**20)
plt.yscale("log")
plt.ylabel("MiB состояния, log")
plt.title("Матрица 512x2048: только основные статистики, без весов и градиентов")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 11. Lion: знак сглаженного направления

Перед шагом $u_t=\beta_1m_{t-1}+(1-\beta_1)g_t$;
обновление $w_{t+1}=(1-\eta\lambda)w_t-\eta\operatorname{sign}(u_t)$.
После шага состояние обновляют другим коэффициентом:
$m_t=\beta_2m_{t-1}+(1-\beta_2)g_t$.
Хранят один момент вместо двух; при ненулевых компонентах величина шага каждой
координаты одинакова, а направление определяется знаком.

Знак теряет информацию о величине, поэтому у Lion обычно настраивают меньший шаг,
чем у Adam. Авторы сообщили о применении в Google Search Ads CTR;
это задокументированный промышленный пример, не универсальное преимущество.
Ниже реализованы сами уравнения; это не вызов неизвестного стороннего пакета.

In [ ]:
class Lion(torch.optim.Optimizer):
    """Учебная реализация уравнений Lion: один момент и знак смешанного направления."""
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        defaults = {"lr": lr, "betas": betas, "weight_decay": weight_decay}
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            beta1, beta2 = group["betas"]
            for parameter in group["params"]:
                if parameter.grad is None:
                    continue
                gradient = parameter.grad
                state = self.state[parameter]
                if "moment" not in state:
                    state["moment"] = torch.zeros_like(parameter)
                moment = state["moment"]
                direction = beta1 * moment + (1 - beta1) * gradient
                parameter.mul_(1 - group["lr"] * group["weight_decay"])
                parameter.add_(torch.sign(direction), alpha=-group["lr"])
                moment.mul_(beta2)
                moment.add_(gradient, alpha=1 - beta2)
        return loss

In [ ]:
point = nn.Parameter(torch.tensor([3., 2.], dtype=torch.float64))
solver = Lion([point], lr=.03, weight_decay=0)
trace = trace_solver(point, solver, steps=100)
solver_traces["Lion"] = trace

## 12. Muon: матричное преобразование инерционного направления

Muon работает с **матрицей** W∈ℝʳˣᶜ. В используемой реализации PyTorch инерцию записывают как экспоненциальное среднее, а не сумму:

$$B_t=\mu B_{t-1}+(1-\mu)G_t,$$
$$D_t=(1-\mu)G_t+\mu B_t\quad\text{при nesterov=True};\qquad D_t=B_t\quad\text{иначе}.$$

Здесь μ=0.95 и B₀=0. После этого преобразуют D целиком. Для SVD $D=U\Sigma V^T$ точная полярная часть — $UV^T$: ненулевые сингулярные значения заменяются единицами. На практике используют конечное число шагов Newton–Schulz:

$$X_0=\frac{D_t}{\max(\|D_t\|_F,\varepsilon)},\qquad A_k=X_kX_k^T,$$
$$X_{k+1}=aX_k+(bA_k+cA_k^2)X_k.$$

Берём a=3.4445, b=−4.775, c=2.0315, K=5 итераций и ε=10⁻⁷. Если r>c, сначала транспонируют D, после итераций транспонируют результат обратно. PyTorch выполняет эту часть в bfloat16. Поэтому результат Qₜ — **приближение**, его сингулярные значения не обязаны в точности равняться единицам.

При `adjust_lr_fn="original"` шаг зависит от формы матрицы:

$$\eta'=\eta\sqrt{\max(1,r/c)},\qquad W_t=(1-\eta\lambda)W_{t-1}-\eta'Q_t.$$

На поверхностях W=diag(w₀,w₁) имеет размер 2×2, поэтому η′=η. Оба диагональных веса обучаются; остальные элементы равны нулю и получают нулевые градиенты. **Все девять методов получают эту же матрицу**, старт и число обновлений. Это простая двумерная иллюстрация; свойства Muon для полноценных матриц дополнительно видны на общем спектральном опыте 4×3. Сохраняется одна B; матричные произведения требуют временной памяти и вычислений. [PyTorch Muon](https://docs.pytorch.org/docs/2.15/generated/torch.optim.Muon.html).

In [ ]:
# Muon принимает матрицу; свободные координаты - её диагональ.
point = nn.Parameter(torch.diag(torch.tensor([3., 2.], dtype=torch.float64)))
solver = torch.optim.Muon([point], lr=.12, momentum=.95, nesterov=True, weight_decay=0,
                          ns_steps=5, ns_coefficients=(3.4445, -4.775, 2.0315), eps=1e-7,
                          adjust_lr_fn="original")
trace = trace_solver(point, solver, steps=100)
solver_traces["Muon"] = trace
assert trace["positions"].shape == (101, 2)
assert np.isfinite(trace["losses"]).all()
print("Muon: сохранены старт и 100 последующих точек; траектория показана на общем графике ниже.")

### Четыре поверхности и общие опыты

В верхней строке выбирают одну из четырёх функций. Для каждой точки высота — значение этой же функции; справа показаны её линии уровня и loss по шагам.

| Пример | f(w) | Градиент и минимум |
|:--|:--|:--|
| Разные масштабы | $(w_0^2+20w_1^2)/2$ | $(w_0,20w_1)$; минимум (0,0) |
| Узкая долина | $(w_0^2+25w_1^2)/2$ | $(w_0,25w_1)$; минимум (0,0) |
| Пологая поверхность | $(0.02w_0^2+w_1^2)/2$ | $(0.02w_0,w_1)$; минимум (0,0) |
| Изогнутая долина | $(1-w_0)^2+20(w_1-w_0^2)^2$ | минимум (1,1) |

У последней функции обозначим r=w₁−w₀². Тогда по правилу цепочки $\partial f/\partial w_0=2(w_0-1)+40r(-2w_0)=2(w_0-1)-80w_0r$, $\partial f/\partial w_1=40r$. Её 3D-высота показана как ln(1+f), чтобы стенки не закрывали дно; градиенты считаются по исходной f.

На каждой поверхности у всех методов одинаковый старт и число шагов. Learning rate у методов различается и указан в настройках: это иллюстрации механизма, а не подбор победителя. Для узкой долины η_SGD=0.075 даёт множитель 1−25η=−0.875: соседние шаги пересекают дно. На пологой поверхности малый loss ещё не означает близость к минимуму, поэтому отдельно показано расстояние.

В двух нижних строках также показаны ответ всех девяти методов на одну историю заданных градиентов, уменьшение весов при нулевом градиенте данных и спектры обновлений общей матрицы 4×3. Нормирование спектров убирает различие общей длины шага. Набор включённых методов сохраняется при смене поверхности; скрываются и линии, и точки последних шагов.

Если редактор не открывает локальные HTML-ссылки, кнопка **«Открыть HTML в браузере»** под графиком использует путь текущего ноутбука через Python. [Отдельный HTML](outputs/optimizer_details.html).

In [ ]:
from support.optimizer_dashboard import (
    make_settings, trace_matrix_surface, mechanism_traces,
    export_dashboard, display_dashboard,
)

base_rates = {"SGD": .03, "momentum": .03, "Нестеров": .03, "AdaGrad": .3,
              "RMSprop": .03, "Adam": .15, "AdamW": .15, "Lion": .03, "Muon": .12}
settings = make_settings(base_rates, weight_decay=0)

def quadratic(points):
    value = (points[..., 0] ** 2 + 20 * points[..., 1] ** 2) / 2
    return value

def narrow_valley(points):
    value = (points[..., 0] ** 2 + 25 * points[..., 1] ** 2) / 2
    return value

def flat_basin(points):
    value = (.02 * points[..., 0] ** 2 + points[..., 1] ** 2) / 2
    return value

def curved_valley(points):
    residual = points[..., 1] - points[..., 0] ** 2
    value = (1 - points[..., 0]) ** 2 + 20 * residual ** 2
    return value

landscape_examples = [
    {"key": "quadratic", "title": "Разные масштабы: исходные настройки", "objective": quadratic,
     "start": (3., 2.), "minimum": (0., 0.), "steps": 100,
     "bounds": ((-3.5, 3.5), (-2.5, 2.5)), "levels": [.05, .2, .5, 1, 2, 5, 10, 20, 40, 80],
     "rates": base_rates, "description": "f=(w0^2+20w1^2)/2. Один старт (3,2), 100 обновлений, lambda=0."},
    {"key": "narrow", "title": "Узкая долина: движение между склонами", "objective": narrow_valley,
     "start": (3., 2.), "minimum": (0., 0.), "steps": 100,
     "bounds": ((-1.7, 3.5), (-2.3, 2.5)), "levels": [.05, .2, .5, 1, 2, 5, 10, 20, 40, 80],
     "rates": {**base_rates, "SGD": .075, "Adam": .22, "AdamW": .22},
     "description": "f=(w0^2+25w1^2)/2. Один старт (3,2), 100 шагов. SGD пересекает дно из-за отрицательного множителя по w1."},
    {"key": "flat", "title": "Пологая поверхность: медленное движение вдоль дна", "objective": flat_basin,
     "start": (4., 2.), "minimum": (0., 0.), "steps": 100,
     "bounds": ((-1.5, 4.7), (-2., 2.5)), "levels": [.005, .02, .05, .1, .2, .5, 1, 2, 3],
     "rates": {**base_rates, "SGD": .45, "momentum": .45, "Нестеров": .45, "Adam": .22, "AdamW": .22},
     "description": "f=(0.02w0^2+w1^2)/2. Один старт (4,2), 100 шагов. Сопоставьте loss и расстояние до (0,0)."},
    {"key": "curved", "title": "Изогнутая долина: поворот направления", "objective": curved_valley,
     "start": (-1.4, 1.6), "minimum": (1., 1.), "steps": 120, "log_height": True,
     "bounds": ((-1.8, 1.5), (-.4, 2.5)), "levels": [.02, .1, .5, 1, 2, 5, 10, 20, 40, 80],
     "rates": {"SGD": .003, "momentum": .003, "Нестеров": .003, "AdaGrad": .15, "RMSprop": .015,
               "Adam": .07, "AdamW": .07, "Lion": .005, "Muon": .02},
     "description": "f=(1-w0)^2+20(w1-w0^2)^2. Один старт (-1.4,1.6), 120 шагов. Только высота 3D преобразована в ln(1+f)."},
]
for example in landscape_examples:
    example_settings = make_settings(example["rates"], weight_decay=0)
    example["traces"] = trace_matrix_surface(example["objective"], example["start"], example_settings, example["steps"])

mechanisms = mechanism_traces(settings)
dashboard_path = export_dashboard(landscape_examples, mechanisms, "outputs/optimizer_details.html")
dashboard_button = display_dashboard(dashboard_path)

assert np.allclose(solver_traces["Muon"]["positions"], landscape_examples[0]["traces"]["Muon"]["positions"])

### Сколько дополнительной памяти занимает оптимизатор

Пусть P — число обучаемых весов. Для одной матрицы 128×512 имеем P=65 536; один массив float32 занимает 4P=262 144 байт, то есть 0.25 MiB. MiB=2²⁰ байт.

После двух настоящих шагов считаем **сохраняемые тензоры `optimizer.state`**. Сами веса, их градиенты, активации, Python-объекты и временные рабочие массивы в это число не входят.

| Метод | Сохраняемые массивы | Дополнительные байты при float32 |
|:--|:--|:--|
| SGD без momentum | нет | 0 |
| momentum, Нестеров | буфер B | 4P |
| AdaGrad | S и скалярный счётчик | 4P + счётчик |
| RMSprop, momentum=0, centered=False | S и счётчик | 4P + счётчик |
| Adam, AdamW | M, V и счётчик | 8P + счётчик |
| Lion | M | 4P |
| Muon | матричный буфер B | 4P |

AdamW не добавляет третий массив: уменьшение весов выполняется непосредственно. У Нестерова градиент и буфер комбинируют при шаге, но отдельную сохраняемую «точку впереди» не хранят. У Lion нет второго момента. У Muon сохраняется одна матрица, однако Newton–Schulz создаёт временные матрицы X, XXᵀ и промежуточные результаты: одинаковая память состояния с momentum **не означает одинаковую пиковую память или время шага**. Их надо профилировать отдельно с учётом формы r×c, устройства и dtype.

На графике наведите на столбец: показаны реальные имена состояний, формы, типы и число байт.

In [ ]:
from support.optimizer_dashboard import measure_state, display_memory

state_rows = measure_state(settings, shape=(128, 512))
display_memory(state_rows)

## 13. MuonClip: стабилизация query/key

В attention вычисляют оценки $A=QK^T/\sqrt d$. Если максимум слишком велик,
softmax может стать почти вырожденным. Для одного учебного head задают
$s=\min(\tau/\max A,1)$; проекции масштабируют $W_Q\leftarrow\sqrt sW_Q$,
$W_K\leftarrow\sqrt sW_K$. Тогда $A_{new}=sA$ и его положительный максимум
не превышает $\tau$. В реальном методе детали зависят от head и архитектуры.

Это ограничение масштаба проекций, а не обрезание нормы общего градиента.
В техническом отчете Kimi K2 описано его использование совместно с Muon.
Ниже изолирован механизм одной головы; это не полное воспроизведение обучения Kimi.

In [ ]:
# Изолированный механизм QK-Clip; это не обучение Kimi и не полная MuonClip-библиотека.
torch.manual_seed(41)
tokens = torch.randn(6, 8)
weight_q = torch.randn(8, 4) * 4
weight_k = torch.randn(8, 4) * 4
query = tokens @ weight_q
key = tokens @ weight_k
attention_logits = query @ key.T / np.sqrt(4)
threshold = 20.0
maximum = attention_logits.max().item()
scale = min(threshold / max(maximum, 1e-8), 1.0)
clipped_query = tokens @ (weight_q * np.sqrt(scale))
clipped_key = tokens @ (weight_k * np.sqrt(scale))
clipped_logits = clipped_query @ clipped_key.T / np.sqrt(4)
assert clipped_logits.max().item() <= threshold + 1e-4
draw_matrices([attention_logits.numpy(), clipped_logits.numpy()],
              [f"До: max logit = {maximum:.1f}", f"После QK-Clip: max = {clipped_logits.max():.1f}"], annotate=False)
figure, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(np.arange(6), torch.softmax(attention_logits[0], dim=0).numpy())
axes[1].bar(np.arange(6), torch.softmax(clipped_logits[0], dim=0).numpy())
axes[0].set(title="Внимание одного токена: до", xlabel="позиция ключа", ylabel="вероятность", ylim=(0, 1))
axes[1].set(title="После ограничения масштаба QK", xlabel="позиция ключа", ylim=(0, 1))
figure.tight_layout()
plt.show()

## 14. Shampoo: матричное предобусловливание

Для градиента матрицы $G_t$ накапливают статистики по двум осям:
$L_t=\rho L_{t-1}+(1-\rho)G_tG_t^T$,
$R_t=\rho R_{t-1}+(1-\rho)G_t^TG_t$.
Двумерный вариант направления:
$D_t=(L_t+\varepsilon I)^{-1/4}G_t(R_t+\varepsilon I)^{-1/4}$.
В отличие от диагонального масштаба, эти матрицы смешивают координаты каждой оси.
Степень вычисляют через собственное разложение симметричной матрицы.

Показывают один механизм с EMA и стабилизацией малых собственных значений.
Промышленная реализация требует блокирования больших тензоров, расписания
предобусловливания и распределения вычислений. У Meta есть Distributed Shampoo;
это инженерная реализация, не утверждение о применении во всех продуктах Meta.
Дополнительная стоимость и память $O(r^2+c^2)$ могут перевесить экономию итераций.

In [ ]:
def symmetric_matrix_power(matrix, power, epsilon=1e-5):
    eigenvalues, eigenvectors = torch.linalg.eigh(matrix)
    stable_eigenvalues = eigenvalues.clamp_min(epsilon)
    powered = stable_eigenvalues.pow(power)
    result = (eigenvectors * powered[None, :]) @ eigenvectors.T
    return result


torch.manual_seed(51)
gradient_history = []
for step in range(20):
    gradient = torch.randn(4, 3)
    gradient[:, 0] *= 10
    gradient[0] *= 4
    gradient_history.append(gradient)
left_statistic = torch.eye(4) * 1e-3
right_statistic = torch.eye(3) * 1e-3
second_elementwise = torch.zeros(4, 3)
for gradient in gradient_history:
    left_statistic = .9 * left_statistic + .1 * (gradient @ gradient.T)
    right_statistic = .9 * right_statistic + .1 * (gradient.T @ gradient)
    second_elementwise = .9 * second_elementwise + .1 * gradient.square()
last_gradient = gradient_history[-1]
left_preconditioner = symmetric_matrix_power(left_statistic, -.25)
right_preconditioner = symmetric_matrix_power(right_statistic, -.25)
shampoo_direction = left_preconditioner @ last_gradient @ right_preconditioner
diagonal_direction = last_gradient / (second_elementwise.sqrt() + 1e-8)
draw_matrices([last_gradient.numpy(), diagonal_direction.numpy(), shampoo_direction.numpy()],
              ["Текущий G", "Покоординатное масштабирование", "Shampoo: $L^{-1/4} G R^{-1/4}$"])
draw_matrices([left_statistic.numpy(), right_statistic.numpy()],
              ["L: взаимодействия строк", "R: взаимодействия столбцов"])

## 15. SOAP: Adam в собственном базисе

Выбирают собственные базисы статистик $Q_L,Q_R$ и преобразуют градиент
$G'_t=Q_L^TG_tQ_R$. В этом базисе накапливают моменты Adam и получают $D'_t$;
обратно переходят как $D_t=Q_LD'_tQ_R^T$.
Так адаптивные диагональные масштабы задаются в выбранных направлениях,
а не только вдоль исходных осей параметров.

Ниже показан шаг в фиксированном базисе. Полный SOAP периодически меняет базис
и согласует с этим состояния; данный код не выдается за распределенный SOAP.
В работе NVIDIA 2026 года и библиотеке Emerging Optimizers исследуют SOAP/Muon
на больших моделях. Это актуальная инженерно-исследовательская практика;
универсальное промышленное доминирование из нее не следует.

In [ ]:
# Один фиксированный базис: показывают принцип шага, не распределенную реализацию SOAP.
_, eigenbasis_left = torch.linalg.eigh(left_statistic)
_, eigenbasis_right = torch.linalg.eigh(right_statistic)
first_rotated = torch.zeros(4, 3)
second_rotated = torch.zeros(4, 3)
for gradient in gradient_history:
    rotated_gradient = eigenbasis_left.T @ gradient @ eigenbasis_right
    first_rotated = .9 * first_rotated + .1 * rotated_gradient
    second_rotated = .99 * second_rotated + .01 * rotated_gradient.square()
steps = len(gradient_history)
first_corrected = first_rotated / (1 - .9 ** steps)
second_corrected = second_rotated / (1 - .99 ** steps)
rotated_update = first_corrected / (second_corrected.sqrt() + 1e-8)
soap_direction = eigenbasis_left @ rotated_update @ eigenbasis_right.T
reconstructed = eigenbasis_left @ rotated_gradient @ eigenbasis_right.T
assert torch.allclose(reconstructed, gradient_history[-1], atol=2e-4)
draw_matrices([rotated_gradient.numpy(), rotated_update.numpy(), soap_direction.numpy()],
              ["G в собственном базисе", "Adam в этом базисе", "Обновление в исходных координатах"])

## 16. LARS и LAMB: относительный шаг слоя

Для направления слоя $d_l$ вычисляют коэффициент порядка
$r_l=\|w_l\|/(\|d_l\|+\varepsilon)$ и обновляют
$w_l\leftarrow w_l-\eta r_ld_l$. Без нулевых норм относительный размер шага
$\|\Delta w_l\|/\|w_l\|$ становится порядка $\eta$.
LARS применяет идею к SGD-подобному направлению, LAMB — к Adam-подобному.
Практические варианты отдельно обрабатывают decay, нулевые нормы, biases и normalization.

Они предназначены прежде всего для больших батчей; результат на батче 128
не проверяет обещания масштабирования на десятки тысяч объектов.
LAMB связан с задокументированным large-batch обучением BERT;
это исторический специализированный метод, а не обязательный новый default.

In [ ]:
# Один большой и один малый слой: одинаковое абсолютное направление не равно одинаковому относительному шагу.
layer_norms = np.array([1., 100.])
direction_norms = np.array([.2, .2])
base_lr = .01
fixed_relative = base_lr * direction_norms / layer_norms
trust_ratio = layer_norms / direction_norms
adapted_relative = base_lr * trust_ratio * direction_norms / layer_norms
figure, axis = plt.subplots(figsize=(8, 3.5))
positions = np.arange(2)
axis.bar(positions - .18, fixed_relative, width=.36, label="один общий eta")
axis.bar(positions + .18, adapted_relative, width=.36, label="eta x ||w слоя|| / ||направления||")
axis.set(xticks=positions, xticklabels=["слой ||w||=1", "слой ||w||=100"],
         ylabel="||Deltaw|| / ||w||", title="Принцип layer-wise trust ratio в LARS/LAMB")
axis.legend(fontsize=8)
figure.tight_layout()
plt.show()

## 17. 8-bit Adam: экономия состояния, а не новый закон градиентного шага

У Adam два основных массива статистик. Их можно хранить в квантованном виде:
разделить на блоки, сохранить коды и масштабы, при обновлении восстановить числа.
У bitsandbytes используется более сложная динамическая блочная схема;
ниже дано учебное линейное int8-квантование, чтобы показать память и ошибку.

Квантование статистик не равно квантованию весов или mixed precision активаций.
Сокращение двух состояний не означает того же сокращения всей памяти обучения.
Практический API семейства — `bitsandbytes.optim.AdamW8bit`; его backend здесь
не запускают и не включают в замер скорости. Никакой универсальный коэффициент
ускорения по учебному рисунку не утверждается.

In [ ]:
# Учебное симметричное блочное int8-квантование; не точная схема bitsandbytes.
generator = np.random.default_rng(61)
state_values = generator.normal(size=1024).astype(np.float32)
state_values[0] = 40
block_size = 64
blocks = state_values.reshape(-1, block_size)
scales = np.max(np.abs(blocks), axis=1, keepdims=True) / 127
codes = np.rint(blocks / scales).clip(-127, 127).astype(np.int8)
restored_states = (codes.astype(np.float32) * scales).reshape(-1)
full_bytes = state_values.nbytes
quantized_bytes = codes.nbytes + scales.nbytes
figure, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(state_values[:100], label="float32")
axes[0].plot(restored_states[:100], linestyle="--", label="int8 + scale")
axes[0].set(title="Большой выброс изолирован своим блоком", xlabel="элемент состояния", ylabel="значение")
axes[0].legend()
axes[1].bar(["float32", "int8 + масштабы"], [full_bytes, quantized_bytes])
axes[1].set(ylabel="байты одной статистики", title="Уменьшается состояние, не вся память модели")
figure.tight_layout()
plt.show()
print("Максимальная абсолютная ошибка:", np.max(np.abs(state_values - restored_states)))

## 18. Schedule-free: разные точки обучения и оценки

Базовая SGD-схема использует три последовательности:
$y_t=(1-\beta)z_t+\beta x_t$,
$z_{t+1}=z_t-\eta\nabla f(y_t)$,
$x_{t+1}=(1-1/(t+1))x_t+z_{t+1}/(t+1)$.
Градиент считают в $y$, оценивают усредненную $x$; постоянный базовый шаг
не требует заранее заданного конца cosine-расписания.

У Meta есть открытая библиотека schedule_free, в том числе AdamW-вариант.
Он требует переключать также `optimizer.train()` / `optimizer.eval()`, а не только модель;
с BatchNorm нужно учитывать статистики. Ниже показана базовая SGD-схема,
не полная реализация AdamWScheduleFree. Это дополнительный метод, не гарантия победы.

In [ ]:
# Базовая schedule-free схема SGD из трех последовательностей, не полная AdamW-версия.
z_sequence = torch.tensor([3., 2.], dtype=torch.float64)
x_sequence = z_sequence.clone()
beta = .9
constant_step = .04
training_points = []
evaluation_points = []
for step in range(100):
    y_sequence = (1 - beta) * z_sequence + beta * x_sequence
    y_sequence.requires_grad_(True)
    objective = optimizer_objective(y_sequence)
    gradient = torch.autograd.grad(objective, y_sequence)[0]
    z_sequence = z_sequence - constant_step * gradient
    averaging_weight = 1 / (step + 1)
    x_sequence = (1 - averaging_weight) * x_sequence + averaging_weight * z_sequence
    training_points.append(y_sequence.detach().numpy().copy())
    evaluation_points.append(x_sequence.detach().numpy().copy())
training_points = np.array(training_points)
evaluation_points = np.array(evaluation_points)
figure, axes = plt.subplots(1, 2, figsize=(11, 3.5))
# Линии уровня общей функции потерь под обеими последовательностями.
sf_w0 = np.linspace(-.3, 3.2, 200)
sf_w1 = np.linspace(-.4, 2.2, 200)
sf_grid_w0, sf_grid_w1 = np.meshgrid(sf_w0, sf_w1)
sf_surface = (sf_grid_w0 ** 2 + 20 * sf_grid_w1 ** 2) / 2
sf_levels = axes[0].contour(
    sf_grid_w0, sf_grid_w1, sf_surface,
    levels=[.02, .1, .5, 1, 2, 5, 10, 20, 40],
    colors="#c3c8cd", linewidths=.8, zorder=0,
)
axes[0].clabel(sf_levels, levels=[.1, 1, 10], fmt="%g", fontsize=7, colors="#7b838c")
axes[0].set_aspect("equal", adjustable="box")
axes[0].scatter(0, 0, color="black", marker="*", s=70, label="минимум", zorder=3)

axes[0].plot(training_points[:, 0], training_points[:, 1], label="y: точка градиента")
axes[0].plot(evaluation_points[:, 0], evaluation_points[:, 1], label="x: точка оценки")
axes[0].set(xlabel="w0", ylabel="w1", title="Обучают и оценивают разные последовательности")
axes[0].legend(fontsize=8)
evaluation_losses = (evaluation_points[:, 0] ** 2 + 20 * evaluation_points[:, 1] ** 2) / 2
axes[1].semilogy(evaluation_losses)
axes[1].set(xlabel="шаг", ylabel="f(x), log", title="Постоянный базовый eta, без cosine")
figure.tight_layout()
plt.show()

## 19. Сравнение с одинаковым бюджетом настройки

В иллюстрациях выше шаги выбирались для демонстрации механизма. Теперь каждому
из девяти методов дают три заранее заданных кандидата и одинаковые 120 обновлений.
Шаг выбирают на одном потоке шума, затем проверяют на трех других seed.
Две геометрии имеют те же собственные значения Hessian, но разные собственные векторы.
К полному градиенту добавляют одинаковые для методов случайные векторы со средним ноль.

Критерий — среднее истинной функции за последние 20 шагов; план настройки фиксирован
до контрольных запусков. Это синтетический опыт об оптимизации, не качество CIFAR-10.
Три кандидата — ограниченный поиск: нельзя из него заключить, что проигравший метод
невозможно настроить лучше. При decay=0 Adam и AdamW здесь служат проверкой эквивалентности.

In [ ]:
# У каждого метода ровно три кандидата eta. Диапазоны заданы до контрольных запусков.
toy_grid = {
    "SGD": [.01, .03, .08], "Momentum": [.005, .015, .04],
    "Nesterov": [.005, .015, .04], "AdaGrad": [.1, .3, 1.0],
    "RMSprop": [.01, .03, .1], "Adam": [.03, .1, .3],
    "AdamW": [.03, .1, .3], "Lion": [.01, .03, .1], "Muon": [.03, .1, .3],
}


def make_toy_solver(name, point, learning_rate):
    if name == "SGD":
        solver = torch.optim.SGD([point], lr=learning_rate)
    elif name == "Momentum":
        solver = torch.optim.SGD([point], lr=learning_rate, momentum=.9)
    elif name == "Nesterov":
        solver = torch.optim.SGD([point], lr=learning_rate, momentum=.9, nesterov=True)
    elif name == "AdaGrad":
        solver = torch.optim.Adagrad([point], lr=learning_rate)
    elif name == "RMSprop":
        solver = torch.optim.RMSprop([point], lr=learning_rate, alpha=.9)
    elif name == "Muon":
        solver = torch.optim.Muon([point], lr=learning_rate, weight_decay=0, adjust_lr_fn="original")
    elif name == "Lion":
        solver = Lion([point], lr=learning_rate, weight_decay=0)
    elif name == "AdamW":
        solver = torch.optim.AdamW([point], lr=learning_rate, weight_decay=0)
    else:
        solver = torch.optim.Adam([point], lr=learning_rate)
    return solver


toy_rows = []
control_traces = {}
toy_selected_rates = {}
for case, rotation in [("оси совпадают", 0.), ("оси повернуты", np.pi / 4)]:
    for name, rates in toy_grid.items():
        candidate_scores = []
        for rate in rates:
            point = nn.Parameter(torch.diag(torch.tensor([3., 2.], dtype=torch.float64)))
            solver = make_toy_solver(name, point, rate)
            trace = trace_solver(point, solver, steps=120, rotation=rotation, noise_std=.3, seed=7)
            score = trace["losses"][-20:].mean() if len(trace["losses"]) == 121 else float("inf")
            candidate_scores.append(score)
        best_index = int(np.argmin(candidate_scores))
        selected_rate = rates[best_index]
        toy_selected_rates[(case, name)] = selected_rate
        for seed in [20, 21, 22]:
            point = nn.Parameter(torch.diag(torch.tensor([3., 2.], dtype=torch.float64)))
            solver = make_toy_solver(name, point, selected_rate)
            trace = trace_solver(point, solver, steps=120, rotation=rotation, noise_std=.3, seed=seed)
            control_traces[(case, name, seed)] = trace
            toy_rows.append({"геометрия": case, "метод": name, "seed": seed,
                             "выбранный eta": selected_rate,
                             "среднее f за 20 последних шагов": trace["losses"][-20:].mean(),
                             "время, с": trace["seconds"]})
toy_results = pd.DataFrame(toy_rows)
toy_summary = toy_results.groupby(["геометрия", "метод"], sort=False).agg(
    learning_rate=("выбранный eta", "first"), mean_final_loss=("среднее f за 20 последних шагов", "mean"),
    std_final_loss=("среднее f за 20 последних шагов", "std"), seconds=("время, с", "mean"))
show_table(toy_summary)
figure, axes = plt.subplots(2, 2, figsize=(15, 9))
for column, case in enumerate(["оси совпадают", "оси повернуты"]):
    summary = toy_summary.loc[case]
    axes[0, column].bar(summary.index, summary.mean_final_loss, yerr=summary.std_final_loss, capsize=3)
    axes[0, column].set(title=case + ": три контрольных seed", ylabel="f, среднее последних 20 шагов")
    axes[0, column].tick_params(axis="x", rotation=50)
    for name in toy_grid:
        curves = [control_traces[(case, name, seed)]["losses"] for seed in [20, 21, 22]]
        mean_curve = np.mean(curves, axis=0)
        axes[1, column].semilogy(mean_curve, label=name)
    axes[1, column].set(xlabel="шаг", ylabel="средняя f, log", title="Одинаковый поток шума для всех методов")
    # Отдельное точное попадание в ноль не должно скрывать остальные кривые.
    axes[1, column].set_ylim(bottom=1e-5)
    axes[1, column].set_ylabel("средняя f, log; нижняя граница 1e-5")
    axes[1, column].legend(fontsize=8)
    axes[1, column].grid(alpha=.2)
figure.tight_layout()
plt.show()
for case in ["оси совпадают", "оси повернуты"]:
    best = toy_summary.loc[case].mean_final_loss.idxmin()
    print(case, ": минимальное среднее в этом опыте -", best)

## 20. Контролируемое сравнение десяти конфигураций на CIFAR-10

Тяжелый опыт включают переменной `W11_OPT_CIFAR=1` до запуска ядра.
Для каждого метода используют ту же CNN и то же разбиение: 12 000 train,
2 000 validation и 2 000 test, восемь эпох, батч 128. Аугментации, BatchNorm,
Dropout и scheduler выключены; weight_decay=0 **для всех**.
Начальные веса и порядок объектов совпадают во всех тридцати запусках.

Каждый метод получает три кандидата своего параметра `lr`. Для Adafactor
семантика этого аргумента иная; равное числовое значение не требовалось.
Сначала выбирают шаг и эпоху по validation log loss, затем один раз оценивают
всех финалистов на test. Не переключают гиперпараметры ради test-accuracy.
Одни методы могут проиграть из-за короткого бюджета или недостаточного диапазона.

`Muon + AdamW` — отдельный **гибрид**: Muon обновляет только скрытую матрицу
`head.2.weight`; четырехмерные свертки, biases и выходной классификатор — AdamW.
Это не опыт «чистый Muon против AdamW для всех параметров».
Сравнение одного seed не является надежным межзадачным рейтингом.

In [ ]:
# Тяжелый опыт включают до запуска ядра: W11_OPT_CIFAR=1.
# При выключении остаются все математические примеры и визуализации механизмов.
RUN_CIFAR_SOLVERS = os.environ.get("W11_OPT_CIFAR", "0") == "1"
OPT_DEVICE = choose_device()
OPT_SEED = 42
OPT_TRAIN = int(os.environ.get("W11_OPT_TRAIN", 12000))
OPT_VALIDATION = int(os.environ.get("W11_OPT_VAL", 2000))
OPT_TEST = int(os.environ.get("W11_OPT_TEST", 2000))
OPT_EPOCHS = int(os.environ.get("W11_OPT_EPOCHS", 8))
OPT_BATCH = 128
CIFAR_SOLVER_GRID = {
    "SGD": [.01, .03, .1], "Momentum": [.01, .03, .1],
    "Nesterov": [.01, .03, .1], "AdaGrad": [.003, .01, .03],
    "RMSprop": [.0003, .001, .003], "Adam": [.0003, .001, .003],
    "AdamW": [.0003, .001, .003], "Lion": [.0001, .0003, .001],
    "Adafactor": [.003, .01, .03], "Muon + AdamW": [.0003, .001, .003],
}
print("CIFAR-поиск включен:", RUN_CIFAR_SOLVERS, "; устройство:", OPT_DEVICE)
print("Кандидатов на метод:", {name: len(rates) for name, rates in CIFAR_SOLVER_GRID.items()})

In [ ]:
class OptimizerConvNet(nn.Module):
    def __init__(self, batch_norm: bool = False, dropout: float = 0.0):
        super().__init__()
        blocks = []
        channels = [3, 16, 32, 64]
        for index in range(3):
            convolution = nn.Conv2d(channels[index], channels[index + 1], 3,
                                    padding=1, bias=not batch_norm)
            blocks.append(convolution)
            if batch_norm:
                blocks.append(nn.BatchNorm2d(channels[index + 1]))
            blocks.append(nn.ReLU())
            if index < 2:
                blocks.append(nn.MaxPool2d(2))
        self.features = nn.Sequential(*blocks)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 10),
        )

    # images: float32 (N, 3, 32, 32); logits: (N, 10).
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        features = self.features(images)
        logits = self.head(features)
        return logits

### Конструкторы и один общий цикл

Оптимизаторы получают параметры одной модели; у гибрида группы не пересекаются.
Учитывают последний неполный батч, validation выполняют в eval-режиме без autograd.
Время включает train, validation и копирование лучшего состояния, но не загрузку данных.
Восстановление весов выбирает лучшую эпоху, а не состояние последнего шага.

In [ ]:
def make_cifar_solvers(name, model, learning_rate):
    parameters = list(model.parameters())
    if name == "SGD":
        solver = torch.optim.SGD(parameters, lr=learning_rate)
    elif name == "Momentum":
        solver = torch.optim.SGD(parameters, lr=learning_rate, momentum=.9)
    elif name == "Nesterov":
        solver = torch.optim.SGD(parameters, lr=learning_rate, momentum=.9, nesterov=True)
    elif name == "AdaGrad":
        solver = torch.optim.Adagrad(parameters, lr=learning_rate)
    elif name == "RMSprop":
        solver = torch.optim.RMSprop(parameters, lr=learning_rate, alpha=.99)
    elif name == "AdamW":
        solver = torch.optim.AdamW(parameters, lr=learning_rate, weight_decay=0)
    elif name == "Lion":
        solver = Lion(parameters, lr=learning_rate, weight_decay=0)
    elif name == "Adafactor":
        solver = torch.optim.Adafactor(parameters, lr=learning_rate, weight_decay=0)
    elif name == "Muon + AdamW":
        # Только скрытая 2D-матрица; Conv2d, biases и выходной классификатор - AdamW.
        matrix_parameters = [model.head[2].weight]
        selected_ids = {id(parameter) for parameter in matrix_parameters}
        other_parameters = [parameter for parameter in parameters if id(parameter) not in selected_ids]
        muon_solver = torch.optim.Muon(matrix_parameters, lr=learning_rate, momentum=.95,
                                       weight_decay=0, adjust_lr_fn="match_rms_adamw")
        other_solver = torch.optim.AdamW(other_parameters, lr=learning_rate, weight_decay=0)
        solvers = [muon_solver, other_solver]
        return solvers
    else:
        solver = torch.optim.Adam(parameters, lr=learning_rate)
    solvers = [solver]
    return solvers


def optimizer_state_bytes(solvers):
    size = 0
    for solver in solvers:
        for state in solver.state.values():
            for value in state.values():
                if torch.is_tensor(value):
                    size += value.numel() * value.element_size()
    return size


def fit_cifar_solver(model, train_loader, validation_loader, solvers, epochs, device):
    criterion = nn.CrossEntropyLoss()
    best_loss = float("inf")
    best_state = None
    history = []
    synchronize(device)
    started = time.perf_counter()
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_sum = 0.0
        count = 0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            for solver in solvers:
                solver.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            for solver in solvers:
                solver.step()
            train_loss_sum += loss.detach().item() * len(labels)
            count += len(labels)
        validation_probabilities = predict_probabilities(model, validation_loader, device)
        validation_labels = validation_loader.dataset.labels.numpy()
        metrics = classification_metrics(validation_labels, validation_probabilities)
        row = {"epoch": epoch, "train_loss": train_loss_sum / count,
               "val_loss": metrics["log_loss"], "val_accuracy": metrics["accuracy"],
               "elapsed_s": time.perf_counter() - started}
        history.append(row)
        if metrics["log_loss"] < best_loss:
            best_loss = metrics["log_loss"]
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
    model.load_state_dict(best_state)
    model.eval()
    synchronize(device)
    elapsed = time.perf_counter() - started
    return history, elapsed

### Тридцать попыток подбора, test еще не используют

Сохраняют все validation-результаты, не только выигравший запуск.
`search_wall_s` включает весь поиск метода, а `train_s` — только его выбранный запуск.
Это разные затраты, их показывают раздельно.

In [ ]:
optimizer_finalists = {}
optimizer_trials = []
if RUN_CIFAR_SOLVERS:
    optimizer_data = load_data(OPT_TRAIN, OPT_VALIDATION, OPT_TEST, seed=OPT_SEED)
    optimizer_mean, optimizer_std = channel_statistics(optimizer_data["train_images"])
    optimizer_train_dataset = ImageDataset(optimizer_data["train_images"], optimizer_data["train_labels"], optimizer_mean, optimizer_std)
    optimizer_validation_dataset = ImageDataset(optimizer_data["val_images"], optimizer_data["val_labels"], optimizer_mean, optimizer_std)
    optimizer_test_dataset = ImageDataset(optimizer_data["test_images"], optimizer_data["test_labels"], optimizer_mean, optimizer_std)
    optimizer_validation_loader = make_loader(optimizer_validation_dataset, OPT_BATCH)
    optimizer_test_loader = make_loader(optimizer_test_dataset, OPT_BATCH)
    for name, rates in CIFAR_SOLVER_GRID.items():
        best_trial = None
        best_score = float("inf")
        synchronize(OPT_DEVICE)
        search_started = time.perf_counter()
        for rate in rates:
            seed_everything(OPT_SEED)
            model = OptimizerConvNet().to(OPT_DEVICE)
            train_loader = make_loader(optimizer_train_dataset, OPT_BATCH, shuffle=True, seed=OPT_SEED)
            solvers = make_cifar_solvers(name, model, rate)
            history, elapsed = fit_cifar_solver(model, train_loader, optimizer_validation_loader, solvers, OPT_EPOCHS, OPT_DEVICE)
            score = min(row["val_loss"] for row in history)
            state_mib = optimizer_state_bytes(solvers) / 2**20
            trial = {"method": name, "lr": rate, "validation_loss": score, "train_s": elapsed,
                     "state_MiB": state_mib, "history": history}
            optimizer_trials.append(trial)
            print(name, "lr", rate, "validation loss", round(score, 4), "seconds", round(elapsed, 2), flush=True)
            if score < best_score:
                best_score = score
                best_trial = {**trial, "weights": {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}}
            del solvers, model
        synchronize(OPT_DEVICE)
        best_trial["search_wall_s"] = time.perf_counter() - search_started
        optimizer_finalists[name] = best_trial
    print("Все настройки выбраны по validation. Теперь разрешена итоговая оценка test.")
else:
    print("Пропущено обучение CNN: для полного опыта задать W11_OPT_CIFAR=1 до запуска ядра.")

### Качество, секунды и память финалистов

Используют одну функцию метрик и один код измерения forward после прогрева.
Учитывают accuracy, macro-F1, balanced accuracy, top-3, log loss и Brier.
Память `state_MiB` — сумма размеров реально выделенных тензоров состояния
в конце обучения, не вся память процесса и не пиковая временная память Muon/Shampoo.
У одной архитектуры число параметров одинаково, а инференс не должен систематически
ускоряться от смены оптимизатора. Различия этих микрозамеров сами по себе не доказывают
эффект оптимизатора и требуют повторения в стабильных условиях.

In [ ]:
if RUN_CIFAR_SOLVERS:
    optimizer_test_rows = []
    optimizer_probabilities = {}
    for name, trial in optimizer_finalists.items():
        model = OptimizerConvNet().to(OPT_DEVICE)
        model.load_state_dict(trial["weights"])
        row, probabilities = evaluate_model(name, model, optimizer_test_loader, OPT_DEVICE, trial["train_s"])
        row.update({"selected_lr": trial["lr"], "state_MiB": trial["state_MiB"],
                    "search_wall_s": trial["search_wall_s"]})
        optimizer_test_rows.append(row)
        optimizer_probabilities[name] = probabilities
    optimizer_comparison = pd.DataFrame(optimizer_test_rows).set_index("model")
    show_table(optimizer_comparison)
    plot_metric_panels(optimizer_comparison, title="CIFAR-10: один seed, один CNN, три пробы на метод")
    plot_metric_panels(optimizer_comparison, columns=["train_s", "state_MiB", "search_wall_s", "batch1_ms", "images_s"],
                       title="Время выбранного запуска, память состояния и весь бюджет поиска")
    selected_histories = {name: trial["history"] for name, trial in optimizer_finalists.items()}
    plot_histories(selected_histories)
    families = [list(selected_histories)[start:start + 4]
                for start in range(0, len(selected_histories), 4)]
    figure, axes = plt.subplots(1, len(families), figsize=(16, 4.5),
                                sharex=True, sharey=True, squeeze=False, layout="constrained")
    from support.lesson11_visuals import PALETTE
    for axis, names in zip(axes.flat, families):
        for name in names:
            frame = pd.DataFrame(selected_histories[name])
            color = PALETTE[list(selected_histories).index(name) % len(PALETTE)]
            axis.plot(frame.elapsed_s, frame.val_loss, marker="o", ms=3, label=name, color=color)
        axis.set(xlabel="Время выбранного запуска, с", ylabel="Validation log loss")
        axis.legend(fontsize=9)
        axis.grid(axis="y")
    figure.suptitle("Одинаковые эпохи требуют разного времени  |  общие шкалы всех панелей")
    plt.show()

    # У одной архитектуры параметров поровну; различается состояние оптимизатора.
    assert optimizer_comparison.parameters.nunique() == 1
    optimizer_output = Path.home() / ".cache" / "ml_dafe" / "week11" / "optimizer_search"
    optimizer_output.mkdir(parents=True, exist_ok=True)
    optimizer_comparison.to_csv(optimizer_output / "comparison.csv")
    pd.DataFrame(optimizer_trials).drop(columns="history").to_csv(optimizer_output / "trials.csv", index=False)
    for name, trial in optimizer_finalists.items():
        torch.save(trial["weights"], optimizer_output / (name.replace(" ", "_") + ".pt"))
    import json
    import platform
    metadata = {"train": OPT_TRAIN, "validation": OPT_VALIDATION, "test": OPT_TEST,
                "epochs": OPT_EPOCHS, "batch": OPT_BATCH, "seed": OPT_SEED,
                "torch": torch.__version__, "device": str(OPT_DEVICE), "platform": platform.platform(),
                "grid": CIFAR_SOLVER_GRID, "weight_decay": 0,
                "muon_scope": "head.2.weight only; other tensors AdamW",
                "mean": optimizer_mean.tolist(), "std": optimizer_std.tolist()}
    (optimizer_output / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
    print("Сохранены результаты:", optimizer_output)

## 21. Как формулировать вывод «кто лучше»

- на графике функции сравнивают скорость оптимизации при указанном бюджете и геометрии;
- на CIFAR-10 сравнивают качество именно этой CNN, выбранное по validation, при одном seed;
- по `train_s` сравнивают выбранные запуски, по `search_wall_s` — всю стоимость подбора;
- память Adam/Lion/Adafactor/Muon сравнивают по состоянию, а не по одинаковым весам модели;
- результаты Kimi, BERT или SOAP нельзя переносить на эту CNN как готовую гарантию.

Практические отправные точки: SGD с momentum/Nesterov для настроенных CNN-рецептов;
AdamW для многих Transformer-рецептов; Adafactor или 8-bit состояния при ограничении
памяти; Muon/Soap/Shampoo — при подходящих матрицах, инфраструктуре и собственном
измерении стоимости. Lion и schedule-free требуют собственных проверок шага и режима.
Это выбор начального эксперимента, а не утверждение о единственном лучшем методе.

**Основания актуального обзора на 13 ноября 2026 года:**

- [API оптимизаторов PyTorch](https://docs.pytorch.org/docs/2.15/optim.html),
  [классификационные рецепты torchvision](https://github.com/pytorch/vision/tree/main/references/classification);
- [AdamW и weight decay](https://arxiv.org/abs/1711.05101),
  [технический отчет DeepSeek-V3](https://arxiv.org/abs/2412.19437);
- [Adafactor](https://arxiv.org/abs/1804.04235), [T5](https://arxiv.org/abs/1910.10683);
- [Lion и описанное авторами production-применение](https://arxiv.org/abs/2302.06675);
- [Muon/Moonlight](https://arxiv.org/abs/2502.16982),
  [MuonClip/Kimi K2](https://www.kimi.ai/blog/kimi-k2);
- [Distributed Shampoo от Meta](https://github.com/facebookresearch/optimizers/tree/main/distributed_shampoo),
  [SOAP](https://arxiv.org/abs/2409.11321),
  [исследование NVIDIA 2026 года](https://arxiv.org/abs/2607.20548),
  [Emerging Optimizers](https://github.com/NVIDIA-NeMo/Emerging-Optimizers);
- [LAMB](https://arxiv.org/abs/1904.00962),
  [8-bit состояния](https://huggingface.co/docs/bitsandbytes/en/explanations/optimizers),
  [schedule-free от Meta](https://github.com/facebookresearch/schedule_free).

Наличие статьи, reference-кода, промышленного применения и массовой распространенности —
разные утверждения. Здесь приводятся конкретные подтвержденные примеры, а не доли рынка.

In [ ]:
# Нет единственного ранга без выбора критерия: строят отдельный ранг каждого столбца.
if RUN_CIFAR_SOLVERS:
    ranking_columns = ["accuracy", "macro_f1", "log_loss", "train_s", "state_MiB"]
    ranks = pd.DataFrame(index=optimizer_comparison.index)
    for metric in ranking_columns:
        descending = metric in ["accuracy", "macro_f1"]
        ranks[metric] = optimizer_comparison[metric].rank(ascending=not descending, method="min")
    figure, axis = plt.subplots(figsize=(9, 6))
    image = axis.imshow(ranks.to_numpy(), cmap="Blues_r", vmin=1, vmax=len(ranks))
    axis.set(xticks=np.arange(len(ranking_columns)), yticks=np.arange(len(ranks)),
             xticklabels=ranking_columns, yticklabels=ranks.index,
             title="Разные критерии - разные лидеры; меньший ранг лучше")
    plt.setp(axis.get_xticklabels(), rotation=35, ha="right")
    for (row, column), value in np.ndenumerate(ranks.to_numpy()):
        axis.text(column, row, f"{value:.0f}", ha="center", va="center", color="white" if value <= 4 else "black")
    figure.colorbar(image, ax=axis, label="ранг внутри этого опыта")
    figure.tight_layout()
    plt.show()

    # Диапазон из трех ставок ограничен: лучший кандидат может лежать на его границе.
    trial_scores = pd.DataFrame(optimizer_trials)
    score_matrix = []
    figure, axis = plt.subplots(figsize=(8, 7))
    for name, rates in CIFAR_SOLVER_GRID.items():
        scores = []
        selected = trial_scores[trial_scores.method == name]
        for rate in rates:
            scores.append(selected[np.isclose(selected.lr, rate)].validation_loss.iloc[0])
        score_matrix.append(scores)
    score_matrix = np.array(score_matrix)
    image = axis.imshow(score_matrix, cmap="Blues_r", aspect="auto")
    for row, (name, rates) in enumerate(CIFAR_SOLVER_GRID.items()):
        best_column = int(np.argmin(score_matrix[row]))
        for column, rate in enumerate(rates):
            axis.text(column, row, f"eta={rate:g}\nL={score_matrix[row, column]:.3f}", ha="center", va="center", fontsize=8,
                      color="white" if score_matrix[row, column] < (score_matrix.min() + score_matrix.max()) / 2 else "black")
        rectangle = plt.Rectangle((best_column - .48, row - .48), .96, .96,
                                  fill=False, edgecolor="#C98A3C", linewidth=2)
        axis.add_patch(rectangle)
    axis.set(xticks=[0, 1, 2], xticklabels=["кандидат 1", "кандидат 2", "кандидат 3"],
             yticks=np.arange(len(CIFAR_SOLVER_GRID)), yticklabels=list(CIFAR_SOLVER_GRID),
             title="Выбор по validation; рамка - выбранный шаг")
    figure.colorbar(image, ax=axis, label="validation log loss: меньше лучше")
    figure.tight_layout()
    plt.show()

    best_accuracy_method = optimizer_comparison.accuracy.idxmax()
    best_probability_method = optimizer_comparison.log_loss.idxmin()
    print("Максимальная test accuracy в этом опыте:", best_accuracy_method)
    print("Минимальный test log loss:", best_probability_method)
    if best_accuracy_method != best_probability_method:
        delta, low, high = paired_accuracy_interval(
            optimizer_data["test_labels"], optimizer_probabilities[best_probability_method],
            optimizer_probabilities[best_accuracy_method], seed=123)
        print(f"Разность accuracy этих фиксированных моделей: {delta:.4f}, парный интервал [{low:.4f}, {high:.4f}]")
    print("Интервал учитывает выборку объектов, но не новые seed обучения и не неопределенность подбора.")

На CIFAR-10 возможна ситуация, когда один метод дает больше верных ответов,
а другой — чуть меньшую кросс-энтропию: вероятность и argmax оценивают разные свойства.
Первое место при кратком поиске не означает оптимальность диапазона; рамка на краю
тепловой карты означает только, что в будущем стоит проверить соседние значения
по validation. Тестовые результаты для выбора нового шага не используют.

Механизм метода объясняет, какие эффекты ожидаются: momentum уменьшает зависимость
направления от одного градиента, Нестеров раньше корректирует инерцию, адаптивные
методы меняют масштабы координат, матричные методы — базис или спектр направления.
Однако причинное объяснение выигрыша конкретной CNN требует отдельных контрольных
опытов. По одной таблице нельзя доказать, что именно этот механизм вызвал прирост.